# Infotheory parity benchmark: NumPy port vs JIDT

Tests every estimator/measure in `pyspi/statistics/infotheory.py` against the JIDT class it replaced.

**Test signal**: bivariate AR(1), unidirectional coupling x->y, 10 seeds per cell.

**T**: {200, 800, 1600}.

**JIDT setup**: matches the historical pyspi `_setup()` (`BIAS_CORRECTION=false`, `NOISE_SEED=42`), JIDT defaults otherwise.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image
pd.set_option('display.float_format', lambda v: f'{v:+.3e}')
df = pd.read_csv('parity_results.csv')
summary = pd.read_csv('parity_summary.csv')
print(f'{len(df)} cells (estimator x measure x T x seed)')


## Summary table


In [ ]:
summary.sort_values(['measure', 'estimator', 'T'])


## Error vs T


In [ ]:
Image('parity_plot.png')


## Findings

### Machine-precision parity (rel_err ~1e-14)
- `gaussian/MI`, `gaussian/TE`, `kernel/MI`, `kernel/entropy`, `symbolic/TE` agree with JIDT to floating-point precision.

### Tiny systematic bias (5e-9 nats, expected)
- `gaussian/entropy`: constant offset of `0.5 * log(1 + 1e-8) = 5e-9` nats from the NumPy port's ridge regularisation `Sigma + 1e-8 * mean(diag) * I`. Deterministic analogue of JIDT's stochastic `NOISE_LEVEL_TO_ADD=1e-8`, documented in `_gaussian_log_det`.

### Convergent finite-sample agreement
- `kraskov/MI`, `kraskov/TE`: KSG-family. Absolute error ~2e-3 nats at T=1600, decreasing with T. Both implementations are unbiased KSG estimators; the residual gap is driven by (i) JIDT's tiny additive observation noise vs NumPy's `eps * (1 - 1e-10)` strict-inequality trick, (ii) `count - 1` vs `count` self-exclusion semantics. Convergence is the expected `O(1/sqrt(N))`.
- `kozachenko/entropy`: <1e-4 nats at T=1600.
- `kernel/TE`: 5e-5 nats at T=1600.

### Bug found + fixed during this benchmark
Initial run showed `kernel/entropy` with a constant ~0.21 nats gap independent of T. Root cause: `KernelEntropyCalculator` standardised the data when `NORMALISE=true` but forgot to add the scale-correction term `d * log2(prod(std))` to the entropy. JIDT instead leaves the data raw and rescales the bandwidth (`kernelWidthsInUse = w * std`), which gives the full entropy with the std term built into the volume `(2*w*std)^d`. The two approaches are equivalent only with that correction.

Fix applied to `pyspi/statistics/infotheory.py::KernelEntropyCalculator.computeAverageLocalOfObservations` — add `+ sum_d log2(std_d)` when normalise=true. Gap drops from +0.21 to 0.00 +/- 0.00 across all T. MI/TE were unaffected because the std factor cancels in the count ratios.

Reference: Kantz & Schreiber, *Nonlinear Time Series Analysis* (1997); Schreiber (2000); Lizier 2014 JIDT paper.

## Convergence expectation

At T=1600, the typical AR(1) MI/TE we're estimating is ~0.03-0.10 nats with KSG estimator standard error of order `1/sqrt(k * N) ~ 0.012`. The implementation-vs-implementation absolute gap we measure (~2e-3 nats) is one order of magnitude below the inherent estimator noise -- the two implementations agree to within ~10% of one estimator standard deviation. T=200 already resolves the patterns; longer T not needed for this question.


# Extended paths: Theiler window, auto-embed, higher embedding

`parity_bench.py` covered defaults (fixed k=1, no Theiler). The shipped configs also use `dyn_corr_excl`, `auto_embed_method=MAX_CORR_AIS`, and higher `k_history`. `parity_bench_extended.py` benchmarks those against JIDT.


In [ ]:
ext = pd.read_csv('parity_summary_extended.csv')
ext.sort_values(['estimator','measure','T'])


## Findings (extended) -- two bugs diagnosed and FIXED

An earlier run of this suite flagged two ported paths that diverged from JIDT. Research (Kraskov et al. 2004; Ragwitz & Kantz 2002; Wibral et al. 2014; JIDT source) confirmed JIDT was correct in both cases, and both were fixed. The numbers below are post-fix.

### Verified clean (always were)
- `gaussian TE_k2` (k_history=2): machine precision (4e-16). The delay-embedding layout in `_te_build_embeddings` is exact for higher order.
- `kraskov TE_k2`, `kraskov TE_DCE5`, `kernel TE_DCE5`: converge to JIDT at the estimator noise floor (~1.5-3e-3 nats at T=1600). The Theiler-windowed TE paths were already correct.

### Fixed -- `kraskov MI_DCE5` (Theiler MI counting)
Before: NumPy gave ~half of JIDT and moved the *wrong direction* (window pushed JIDT's MI up, the port's down). Root cause: the windowed branch of `_ksg_mi_pair` counted marginal neighbours **inclusively** (<= eps) while the w=0 branch and JIDT count **strictly** (< eps); the boundary k-th neighbour inflated n_x/n_y and flipped the sign. KSG1 keeps the full N in psi(N) (only the *neighbour set* is windowed) -- confirmed verbatim in JIDT and IDTxl. Fix: strict marginal counting (`eps*(1-1e-10)`), full N. After: abs_err 1.8e-2 -> **2.3e-3** at T=1600, correct direction, at the KSG noise floor.

### Fixed -- auto-embed (bias-corrected AIS)
Before: `_gaussian_ais` was the raw in-sample log-det multiinformation with no bias correction, so it increased monotonically in k and saturated at `k_search_max` (median k=10). Fix: subtract the chi-squared-null mean `k/(2N)` (df = dim(Y_f)*k = k), matching JIDT's `ActiveInfoStorageCalculatorGaussian`. This yields an interior maximum. After: `gaussian TE_autoembed` selects the same (k,tau) as JIDT in **100%** of seeds/T (median k=7) and converges to ~1.9e-3 at T=1600.

### Also fixed -- MI/TLMI `dyn_corr_excl: AUTO` routing
MI/TLMI previously coerced any string (incl. `AUTO`) to w=0, silently dropping the Theiler window -- a regression from the JIDT-era `_set_theiler_window`. `_resolve_theiler` was lifted to `JIDTBase` and the four MI/TLMI kraskov sites (bivariate + multivariate) now compute the per-pair autocorrelation window. With the counting fix above, AUTO now applies a correct window.

### Remaining approximation -- `kraskov TE_autoembed` (~26% off JIDT)
Now non-saturating, but the port still selects the embedding with **Gaussian** AIS even for the kraskov estimator (picks k=7), whereas JIDT MAX_CORR_AIS uses the destination's **own** (KSG) estimator (picks k=2). JIDT's *gaussian* auto-embed also picks k=7, so the gaussian criterion is faithful; the gap is purely gaussian-AIS-vs-KSG-AIS for selection. No shipped config enables kraskov auto-embedding, so this is left as a documented approximation (estimator-consistent KSG-AIS embedding is a possible future upgrade).

### JIDT limitation -- `symbolic TE_k10`
JIDT throws `ArrayIndexOutOfBoundsException` at k_history=10 (10! symbols overflow its joint-histogram index), so `symbolic k_history=10` could not have run under the original JIDT pyspi. The NumPy port (np.unique over *observed* symbols) does not crash, but at k=10 with T<=1600 the symbol space is massively undersampled, so the values (~1e-3) are finite-sample artifacts. Use smaller k or much larger T.
